## Лабораторная работа № 1
## Разведочный анализ больших данных с Apache Spark

### Часть 2: разведочный анализ данных (EDA)

In [ ]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import col, lit, sum, mean, when, count, desc, floor, corr
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


def create_spark_configuration() -> SparkConf:
    """Cozdaet i konfiguriruet ekzemplyar SparkConf dlya prilozheniya Spark."""
    user_name = os.getenv("USER")
    conf = SparkConf()
    conf.setAppName("SOBD lab1 - NYC Taxi EDA")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    return conf

In [ ]:
conf = create_spark_configuration()
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

In [ ]:
database_name = "shahulov_database"
spark.catalog.setCurrentDatabase(database_name)
df = spark.table("sobd_lab1_table")

In [ ]:
df.show()

In [ ]:
df.printSchema()

In [ ]:
df.count()

#### Пропущенные значения

In [ ]:
def count_nulls(data: DataFrame, column_name: str) -> None:
    """Podschet kolichestva null i not null znachenij v stolbce."""
    null_counts = data.select(sum(col(column_name).isNull().cast("int"))).collect()[0][0]
    not_null_counts = data.select(sum(col(column_name).isNotNull().cast("int"))).collect()[0][0]
    total = (null_counts or 0) + (not_null_counts or 0)
    pct = 100 * (null_counts or 0) / total if total else 0
    print(f"{column_name}: NULL = {null_counts} ({pct:.2f}%)")

In [ ]:
for c in df.columns:
    count_nulls(df, c)

Удаляем явные ошибки и пропуски в ключевых полях.

In [ ]:
df = df.dropna(subset=["trip_distance", "fare_amount", "total_amount", "trip_duration_min"])
df = df.fillna({"passenger_count": 1, "RatecodeID": 1, "payment_type": 5})
df.count()

#### Категориальные признаки

In [ ]:
def plot_cat_distribution(data: DataFrame, column_name: str, top_n: int = 20) -> None:
    """Raspredelenie kategorialnogo priznaka."""
    categories = data.groupBy(column_name).count().orderBy("count", ascending=False)
    print(f"Kolichestvo kategorij {column_name}: {categories.count()}")
    categories = categories.limit(top_n).toPandas()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=column_name, y="count", data=categories)
    plt.title(f"Raspredelenie \"{column_name}\"")
    plt.xlabel(column_name)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plot_cat_distribution(df, "payment_type")

In [ ]:
plot_cat_distribution(df, "passenger_count")

#### Числовые признаки: статистика и выбросы

In [ ]:
def plot_boxplots(data: DataFrame, columns: list, sample_fraction: float = 0.1) -> None:
    """Boxplot i statistiki dlya chislovyh stolbcov."""
    box_data = []
    for column in columns:
        q1, median, q3 = data.approxQuantile(column, [0.25, 0.5, 0.75], 0.01)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outliers_df = data.filter((col(column) < lower_bound) | (col(column) > upper_bound))
        min_value = data.agg({column: "min"}).collect()[0][0]
        mean_value = data.agg({column: "mean"}).collect()[0][0]
        std_value = data.agg({column: "std"}).collect()[0][0]
        max_value = data.agg({column: "max"}).collect()[0][0]
        lower_bound = max(lower_bound, min_value)
        upper_bound = min(upper_bound, max_value)
        outliers = []
        if not outliers_df.isEmpty():
            sampled = outliers_df.sample(sample_fraction).select(column).limit(1000).collect()
            outliers = [r[column] for r in sampled]
        box_data.append({"whislo": lower_bound, "q1": q1, "med": median, "q3": q3, "whishi": upper_bound, "fliers": outliers})
        print(f"--- {column} ---")
        print(f"min={min_value:.2f} mean={mean_value:.2f} std={std_value:.2f} q1={q1:.2f} median={median:.2f} q3={q3:.2f} max={max_value:.2f}")
    fig, ax = plt.subplots(figsize=(20, 6))
    ax.bxp(box_data, vert=False, positions=range(1, len(columns) + 1), widths=0.5)
    ax.set_yticks(range(1, len(columns) + 1))
    ax.set_yticklabels(columns)
    ax.set_xlabel("Value")
    ax.set_title("Boxplots")
    ax.grid(True)
    plt.show()

In [ ]:
def plot_quant_distribution(data: DataFrame, column: str, num_bins: int = 100) -> None:
    """Gistogramma raspredeleniya chislovogo priznaka."""
    min_value = data.agg({column: "min"}).collect()[0][0]
    max_value = data.agg({column: "max"}).collect()[0][0]
    bin_size = (max_value - min_value) / num_bins
    if bin_size == 0:
        print("Stolbec postoyannyj, gistogramma ne stroitsya")
        return
    data = data.withColumn("bin", floor((col(column) - min_value) / bin_size))
    bin_counts = data.groupBy("bin").count().orderBy("bin").toPandas()
    bin_counts["bin_center"] = bin_counts["bin"].apply(lambda x: min_value + (x + 0.5) * bin_size)
    plt.figure(figsize=(20, 6))
    sns.histplot(data=bin_counts, x="bin_center", weights="count", bins=num_bins)
    plt.xlabel("Value")
    plt.ylabel("Count")
    plt.title(f"Raspredelenie \"{column}\"")
    plt.grid(True)
    plt.show()

In [ ]:
plot_boxplots(df, ["trip_distance"])

In [ ]:
plot_quant_distribution(df, "trip_distance")

Обрезаем выбросы и некорректные значения.

In [ ]:
df = df.filter((col("trip_distance") > 0) & (col("trip_distance") < 100))
df = df.filter((col("fare_amount") > 0) & (col("fare_amount") < 500))
df = df.filter((col("total_amount") > 0) & (col("total_amount") < 600))
df = df.filter((col("trip_duration_min") > 0) & (col("trip_duration_min") < 240))
df.count()

In [ ]:
plot_boxplots(df, ["trip_distance", "fare_amount", "total_amount", "trip_duration_min"])

In [ ]:
plot_quant_distribution(df, "fare_amount")

In [ ]:
plot_quant_distribution(df, "trip_duration_min")

#### Корреляции между признаками

In [ ]:
def plot_correlation(data: DataFrame, columns: list) -> None:
    """Matrica korrelyacij Pirsona dlya chislovyh priznakov."""
    clean = data.select(columns).na.drop()
    assembler = VectorAssembler(inputCols=columns, outputCol="features")
    vec = assembler.transform(clean).select("features")
    matrix = Correlation.corr(vec, "features", "pearson").collect()[0][0]
    corr_pd = pd.DataFrame(matrix.toArray(), index=columns, columns=columns)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_pd, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Matrica korrelyacij")
    plt.show()
    return corr_pd

In [ ]:
num_cols = ["trip_distance", "fare_amount", "tip_amount", "tolls_amount", "total_amount", "trip_duration_min", "passenger_count"]
corr_pd = plot_correlation(df, num_cols)
corr_pd

#### Выводы

Заполните выводы по результатам анализа.

In [ ]:
spark.stop()